In [7]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from itertools import combinations
from scipy.stats import ttest_ind
import pandas as pd

Entire the desired output_directory, model_name, and metric we care about. Then run all the code below

In [8]:
output_dir = 'models'

output_dir = Path('/data/vision/polina/users/marcusbl/bin_class/outputs') / output_dir

model_name = 'model_loss'
metrics = ['auc', 'tpr_at_10%', 'tpr_at_20%', 'tpr_at_30%']

Parse results into Pandas DF

In [9]:
results = []

for test_dir in output_dir.iterdir():
    for run_dir in test_dir.iterdir():
        if not run_dir.is_dir():
            continue

        with open(run_dir / 'test_info' / 'metric_info.json') as f:
            metric_info = json.load(f)

        flat = {
            (model, metric): value
            for model, metrics in metric_info.items()
            for metric, value in metrics.items()
        }

        results.append({
            "test": test_dir.name,
            "run": run_dir.name,
            **flat
        })

df = pd.DataFrame(results).set_index(["test", "run"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df = df.sort_index(axis=1).sort_index()


In [10]:
df['model_auc']

acc    auc  epoch     f1  fn   fp    fpr   loss   prec  \
test      run                                                              
convnext  run0  0.858  0.897     -1  0.599  99   47  0.057  0.333  0.699   
          run1  0.893  0.896     -1  0.700  60   49  0.059  0.397  0.722   
          run2  0.916  0.955     -1  0.775  25   53  0.069  0.292  0.717   
          run3  0.902  0.951     -1  0.653  21   82  0.088  0.240  0.542   
          run4  0.926  0.922     -1  0.683  47   33  0.035  0.199  0.723   
          run5  0.911  0.942     -1  0.774  48   52  0.058  0.320  0.767   
          run6  0.904  0.952     -1  0.731  37   77  0.077  0.267  0.668   
          run7  0.869  0.915     -1  0.695  41   98  0.113  0.293  0.617   
          run8  0.915  0.914     -1  0.614  78   20  0.020  0.248  0.796   
          run9  0.939  0.976     -1  0.813  42   22  0.025  0.172  0.863   
resnet101 run0  0.877  0.897     -1  0.652  89   38  0.046  0.347  0.758   
          run1  0.883  0.880     -1  0.625  87   33  0.039  0.339  0.752   
          run2  0.841  0.950     -1  0.667  12  135  0.176  0.381  0.521   
          run3  0.877  0.957     -1  0.620  12  118  0.126  0.304  0.473   
          run4  0.908  0.909     -1  0.615  54   45  0.047  0.225  0.637   
          run5  0.826  0.935     -1  0.666  25  170  0.188  0.371  0.533   
          run6  0.831  0.938     -1  0.634  18  183  0.183  0.383  0.487   
          run7  0.878  0.926     -1  0.689  55   75  0.087  0.266  0.658   
          run8  0.896  0.923     -1  0.641  49   71  0.071  0.268  0.601   
          run9  0.930  0.967     -1  0.786  47   26  0.030  0.168  0.838   
resnet152 run0  0.859  0.902     -1  0.700  39  106  0.129  0.356  0.615   
          run1  0.904  0.903     -1  0.713  65   33  0.039  0.283  0.787   
          run2  0.811  0.946     -1  0.628  11  164  0.214  0.416  0.474   
          run3  0.883  0.944     -1  0.619  18  105  0.112  0.288  0.488   
          run4  0.921  0.926     -1  0.619  64   21  0.022  0.266  0.767   
          run5  0.898  0.927     -1  0.732  62   53  0.059  0.253  0.748   
          run6  0.906  0.948     -1  0.747  27   85  0.085  0.272  0.660   
          run7  0.874  0.932     -1  0.698  44   90  0.104  0.305  0.633   
          run8  0.920  0.944     -1  0.667  64   28  0.028  0.201  0.767   
          run9  0.810  0.973     -1  0.639   5  194  0.224  0.392  0.476   
resnet18  run0  0.887  0.891     -1  0.665  93   23  0.028  0.370  0.833   
          run1  0.865  0.882     -1  0.627  71   67  0.080  0.350  0.634   
          run2  0.869  0.929     -1  0.686  27   94  0.123  0.297  0.584   
          run3  0.809  0.934     -1  0.506  15  186  0.199  0.417  0.356   
          run4  0.909  0.932     -1  0.669  34   64  0.068  0.230  0.607   
          run5  0.876  0.934     -1  0.720  40   99  0.110  0.281  0.644   
          run6  0.914  0.942     -1  0.755  35   67  0.067  0.215  0.701   
          run7  0.845  0.924     -1  0.661  38  127  0.147  0.344  0.559   
          run8  0.842  0.918     -1  0.583  28  155  0.155  0.381  0.452   
          run9  0.901  0.956     -1  0.752  23   81  0.094  0.236  0.661   
resnet34  run0  0.884  0.895     -1  0.706  65   54  0.066  0.305  0.726   
          run1  0.882  0.882     -1  0.611  92   29  0.035  0.319  0.766   
          run2  0.798  0.931     -1  0.610  13  174  0.227  0.514  0.456   
          run3  0.855  0.946     -1  0.579  13  140  0.150  0.327  0.429   
          run4  0.862  0.924     -1  0.607  18  131  0.138  0.352  0.467   
          run5  0.851  0.936     -1  0.696  28  139  0.154  0.307  0.579   
          run6  0.919  0.926     -1  0.734  58   39  0.039  0.227  0.775   
          run7  0.880  0.917     -1  0.700  50   78  0.090  0.283  0.656   
          run8  0.891  0.919     -1  0.644  42   84  0.084  0.257  0.576   
          run9  0.901  0.967     -1  0.761  15   89  0.103  0.247  0.651   
resnet50  run0  0.867  0.892     -1  0.662  74   63  0.077  0.316 

Average/Variance for desired metrics over all runs for the desired model

In [11]:
m = df[model_name]

stats = m.groupby(level='test').agg(['mean', 'var'])

stats = stats.sort_values(by=('auc', 'mean'), ascending=False)

stats[[(metric, stat) for metric in metrics for stat in ['mean', 'var']]]

auc           tpr_at_10%           tpr_at_20%            \
             mean       var       mean       var       mean       var   
test                                                                    
resnet50   0.9343  0.000712     0.8122  0.003728     0.9074  0.002350   
convnext   0.9301  0.000843     0.8045  0.005202     0.8975  0.002170   
resnet152  0.9295  0.000569     0.8127  0.001831     0.8980  0.001158   
resnet101  0.9268  0.000740     0.8018  0.003140     0.8936  0.001854   
resnet18   0.9205  0.000413     0.7796  0.002946     0.8753  0.001818   
resnet34   0.9169  0.000763     0.7705  0.003622     0.8714  0.002360   

          tpr_at_30%            
                mean       var  
test                            
resnet50      0.9398  0.001432  
convnext      0.9326  0.001598  
resnet152     0.9370  0.001288  
resnet101     0.9346  0.001680  
resnet18      0.9202  0.000940  
resnet34      0.9127  0.001441

Paired T-test for signficance. Note that it's paired b/c same seed for each 

In [16]:
from scipy.stats import ttest_rel
import pandas as pd
import numpy as np

rows = []

tests = m.index.get_level_values("test").unique()

for metric in metrics:
    # Pull each test's series once
    series_by_test = {}
    for test in tests:
        s = m.loc[test, metric]
        if isinstance(s, pd.DataFrame):
            raise ValueError(f"Expected a Series for test={test}, metric={metric}, got DataFrame.")
        series_by_test[test] = s.dropna()

    # Mean over available runs for ranking
    mean_per_test = pd.Series({
        test: s.mean() for test, s in series_by_test.items() if len(s) > 0
    })

    if mean_per_test.empty:
        continue

    # Best test for this metric
    best_test = mean_per_test.idxmax()
    best_vals = series_by_test[best_test]

    for test in tests:
        if test == best_test:
            continue

        other_vals = series_by_test[test]

        # Align by run/seed so only matching runs are compared
        paired = pd.concat(
            [best_vals.rename("best"), other_vals.rename("other")],
            axis=1,
            join="inner"
        ).dropna()

        n_pairs = len(paired)

        if n_pairs < 2:
            mean_best = np.nan
            mean_test = np.nan
            diff = np.nan
            t_stat = np.nan
            p_value = np.nan
        else:
            mean_best = paired["best"].mean()
            mean_test = paired["other"].mean()
            diff = (paired["best"] - paired["other"]).mean()
            t_stat, p_value = ttest_rel(paired["best"], paired["other"], nan_policy="omit")

        rows.append({
            "metric": metric,
            "best_test": best_test,
            "test": test,
            "mean_best": mean_best,
            "mean_test": mean_test,
            "diff": diff,
            "n_pairs": n_pairs,
            "t_stat": t_stat,
            "p_value": p_value,
            "significant": bool(p_value < 0.05) if pd.notna(p_value) else np.nan,
        })

paired_tstats = (
    pd.DataFrame(rows)
    .set_index(["metric", "best_test", "test"])
    .sort_index()
)

paired_tstats

mean_best  mean_test    diff  n_pairs  \
metric     best_test test                                               
auc        resnet50  convnext      0.9343     0.9301  0.0042       10   
                     resnet101     0.9343     0.9268  0.0075       10   
                     resnet152     0.9343     0.9295  0.0048       10   
                     resnet18      0.9343     0.9205  0.0138       10   
                     resnet34      0.9343     0.9169  0.0174       10   
tpr_at_10% resnet152 convnext      0.8127     0.8045  0.0082       10   
                     resnet101     0.8127     0.8018  0.0109       10   
                     resnet18      0.8127     0.7796  0.0331       10   
                     resnet34      0.8127     0.7705  0.0422       10   
                     resnet50      0.8127     0.8122  0.0005       10   
tpr_at_20% resnet50  convnext      0.9074     0.8975  0.0099       10   
                     resnet101     0.9074     0.8936  0.0138       10   
                     resnet152     0.9074     0.8980  0.0094       10   
                     resnet18      0.9074     0.8753  0.0321       10   
                     resnet34      0.9074     0.8714  0.0360       10   
tpr_at_30% resnet50  convnext      0.9398     0.9326  0.0072       10   
                     resnet101     0.9398     0.9346  0.0052       10   
                     resnet152     0.9398     0.9370  0.0028       10   
                     resnet18      0.9398     0.9202  0.0196       10   
                     resnet34      0.9398     0.9127  0.0271       10   

                                  t_stat   p_value  significant  
metric     best_test test                                        
auc        resnet50  convnext   1.240552  0.246136        False  
                     resnet101  1.679619  0.127336        False  
                     resnet152  2.478342  0.035087         True  
                     resnet18   2.815870  0.020183         True  
                     resnet34   3.170690  0.011353         True  
tpr_at_10% resnet152 convnext   0.532478  0.607289        False  
                     resnet101  0.789595  0.450065        False  
                     resnet18   3.110286  0.012513         True  
                     resnet34   4.202850  0.002297         True  
                     resnet50   0.052381  0.959370        False  
tpr_at_20% resnet50  convnext   1.870590  0.094210        False  
                     resnet101  1.693595  0.124588        False  
                     resnet152  1.344016  0.211840        False  
                     resnet18   2.654908  0.026264         True  
                     resnet34   3.028884  0.014272         True  
tpr_at_30% resnet50  convnext   1.089909  0.304073        False  
                     resnet101  0.831766  0.427058        False  
                     resnet152  0.755684  0.469154        False  
                     resnet18   2.163813  0.058702        False  
                     resnet34   3.654569  0.005280         True